In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Временные ряды

In [ ]:
#подготовка ряда к обработке
PATH = "your_file.csv"
DATE_COL = "date"     # колонка с датой
Y_COL = "y"           # колонка со значением ряда

df = pd.read_csv(PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
ts = df.set_index(DATE_COL)[Y_COL].copy()
ts = ts.astype(float)  #сто проц число
ts.head()

посмотрим основные пропуски в рядах по времени

In [ ]:
# шаг между точками
diffs = ts.index.to_series().diff().value_counts().head(5)
print(diffs)
# частота автоматически
freq = pd.infer_freq(ts.index)
print("infer_freq:", freq)

приводим если надо к одночастотной сетке

In [ ]:
#'D' день, 'W' неделя, 'M' месяц
TARGET_FREQ = "D"
ts_reg = ts.asfreq(TARGET_FREQ)
print("пропуски после сетки:", ts_reg.isna().sum())

визуализация ряда база

In [ ]:
plt.figure()
plt.plot(ts_filled.index, ts_filled.values)
plt.title("Временной ряд (после приведения к регулярной сетке)")
plt.xlabel("Время")
plt.ylabel(Y_COL)
plt.grid(True)
plt.show()

распределение значений и боксплот

In [ ]:
plt.figure()
plt.hist(ts_filled.dropna().values, bins=30)
plt.title("Распределение значений")
plt.xlabel(Y_COL)
plt.ylabel("Частота")
plt.grid(True)
plt.show()

plt.figure()
plt.boxplot(ts_filled.dropna().values, vert=False)
plt.title("Boxplot")
plt.xlabel(Y_COL)
plt.grid(True)
plt.show()

смотрим окна и тренды (база визуализация)

In [ ]:
WINDOW = 30  #окно
roll_mean = ts_filled.rolling(WINDOW).mean()
roll_std = ts_filled.rolling(WINDOW).std()
plt.figure()
plt.plot(ts_filled.index, ts_filled.values, label="y")
plt.plot(roll_mean.index, roll_mean.values, label=f"rolling mean ({WINDOW})")
plt.plot(roll_std.index, roll_std.values, label=f"rolling std ({WINDOW})")
plt.title("Rolling mean / std")
plt.xlabel("Время")
plt.ylabel(Y_COL)
plt.grid(True)
plt.legend()
plt.show()

проверяем сезонность (визуализации)

In [ ]:
#по дню недели
tmp = ts_filled.to_frame("y")
tmp["dow"] = tmp.index.dayofweek  # 0=Mon ... 6=Sun
dow_mean = tmp.groupby("dow")["y"].mean()
plt.figure()
plt.plot(dow_mean.index, dow_mean.values, marker="o")
plt.title("Среднее значение по дню недели (сезонность недели)")
plt.xlabel("dayofweek (0=Mon ... 6=Sun)")
plt.ylabel("mean(y)")
plt.grid(True)
plt.show()

In [ ]:
#по месяцу
tmp["month"] = tmp.index.month
m_mean = tmp.groupby("month")["y"].mean()
plt.figure()
plt.plot(m_mean.index, m_mean.values, marker="o")
plt.title("Среднее значение по месяцу")
plt.xlabel("month")
plt.ylabel("mean(y)")
plt.grid(True)
plt.show()

автокорреляция

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.figure()
plot_acf(ts_filled.dropna(), lags=40)
plt.grid(True)
plt.show()
plt.figure()
plot_pacf(ts_filled.dropna(), lags=40, method="ywm")
plt.grid(True)
plt.show()

треин - тест (сплит по времени)

In [ ]:
H = 30  # на скок прогноз
train = ts_filled.iloc[:-H]
test = ts_filled.iloc[-H:]
print("train:", train.index.min(), "->", train.index.max(), "len=", len(train))
print("test :", test.index.min(), "->", test.index.max(), "len=", len(test))

общий тест на стационарность ряда

Сезонность и структурные сдвиги могут мешать: иногда нужно сезонное дифференцирование (для SARIMA) или сначала убрать сезонность.

ADF-тест гипотезы:

	•	H0 (нулевая): ряд нестационарен (есть единичный корень, unit root)
	•	H1 (альтернатива): ряд стационарен

тогда:

	•	если p-value < 0.05 → отвергаем H0 → ряд стационарен
	•	если p-value >= 0.05 → H0 не отвергаем → ряд нестационарен

In [ ]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

def adf_stationarity_test(series, title=""):
    s = pd.Series(series).dropna()
    result = adfuller(s, autolag="AIC")  # выбирает число лагов автоматически
    stat, pvalue, used_lags, nobs = result[0], result[1], result[2], result[3]
    crit = result[4]  # критические значения
    print(f"ADF тест: {title}")
    print(f"ADF statistic = {stat:.4f}")
    print(f"p-value = {pvalue:.4f}")
    print(f"lags used= {used_lags}")
    print(f"nobs= {nobs}")
    print("  critical values:")
    for k, v in crit.items():
        print(f"    {k}: {v:.4f}")

    if pvalue < 0.05:
        print(" ряд скорее стационарен (H0 отвергаем)\n")
    else:
        print("ряд скорее нестационарен (H0 НЕ отвергаем)\n")

# вызов
adf_stationarity_test(ts)
adf_stationarity_test(ts.diff(1))

 План анализа ARIMA

	1.	Подготовка временного ряда
	2.	EDA-графики (линия, распределение, rolling mean/std)
	3.	Стационарность: тест ADF + решение про d (разности)
	4.	ACF/PACF (подсказки для p и q)
	5.	Разбиение train/test по времени
	6.	Подбор ARIMA(p,d,q) (grid search по AIC)
	7.	Обучение лучшей модели + прогноз + доверительные интервалы
	8.	Метрики качества (MAE, RMSE, MAPE)
	9.	Диагностика остатков (остатки, QQ-plot, ACF остатков)

проверим стационарность

In [ ]:
from statsmodels.tsa.stattools import adfuller
def adf_test(series, title=""):
    series = series.dropna()
    result = adfuller(series, autolag="AIC")
    stat, pvalue, used_lags, nobs = result[0], result[1], result[2], result[3]
    print(f"ADF Test {title}")
    print(f" ADF statistic: {stat:.4f}")
    print(f"p-value: {pvalue:.4f}")
    print(f"lags used: {used_lags}")
    print(f"nobs: {nobs}")
adf_test(ts_filled, "Исходный ряд")
adf_test(ts_filled.diff(1), "Разность d=1")
adf_test(ts_filled.diff(2), "Разность d=2")

	•	если исходный ряд не стационарен (p-value ≥ 0.05), но после diff(1) стал стационарен → берём d = 1
	•	если только после diff(2) → d = 2
	•	если исходный уже стационарен → d = 0

выбор p и q

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
d = 1  # ставим d как на прошлом
series_d = ts_filled.diff(d).dropna()
plt.figure()
plot_acf(series_d, lags=40)
plt.grid(True)
plt.show()
plt.figure()
plot_pacf(series_d, lags=40, method="ywm")
plt.grid(True)
plt.show()

In [ ]:
# смотрим доверительный интервал
model = ARIMA(train, order=best_order).fit()
forecast_res = model.get_forecast(steps=H)
pred = forecast_res.predicted_mean
conf = forecast_res.conf_int(alpha=0.05)  # 95 доверительный интервал
pred.index = test.index
conf.index = test.index
plt.figure()
plt.plot(train.index, train.values, label="train")
plt.plot(test.index, test.values, label="test")
plt.plot(pred.index, pred.values, label="ARIMA forecast")
plt.fill_between(conf.index, conf.iloc[:, 0], conf.iloc[:, 1], alpha=0.3)
plt.title(f"ARIMA{best_order}: forecast + 95% CI")
plt.xlabel("Время")
plt.ylabel(Y_COL)
plt.grid(True)
plt.legend()
plt.show()

# неоднородные временные ряды

In [ ]:
def prepare_ts(df, date_col, y_col, freq=None, fill="time"):
    """
    df
    date_col :дата/время
    y_col    :значение ряда
    freq     : частота "M"
    fill     :
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    ts = df.set_index(date_col)[y_col].astype(float)

    if freq is not None:
        ts = ts.asfreq(freq)

    if fill == "time":
        ts = ts.interpolate(method="time")
    elif fill == "ffill":
        ts = ts.ffill()

    return ts

# пример:
df = pd.read_csv("data.csv")
s = prepare_ts(df, "date", "y", freq="D", fill="time")

In [ ]:
#обноружение
plt.figure()
plt.plot(ts.index, ts.values)
plt.title("Временной ряд")
plt.xlabel("Время")
plt.ylabel(Y_COL)
plt.grid(True)
plt.show()

меры раннеого обноружения

In [ ]:
WINDOW = 200
measures_roll = pd.DataFrame(index=ts.index)
measures_roll["var"]  = ts.rolling(WINDOW).var()
measures_roll["skew"] = ts.rolling(WINDOW).skew()
measures_roll["kurt"] = ts.rolling(WINDOW).kurt()
def _acf1(x):
    x = pd.Series(x).dropna()
    if len(x) < 3:
        return np.nan
    return x.autocorr(lag=1)

measures_roll["acf1"] = ts.rolling(WINDOW).apply(_acf1, raw=False)
measures_roll.tail()

In [ ]:
for col in ["var", "skew", "kurt", "acf1"]:
    plt.figure()
    plt.plot(measures_roll.index, measures_roll[col].values)
    plt.title(f"Rolling measure: {col} (window={WINDOW})")
    plt.xlabel("Время")
    plt.grid(True)
    plt.show()

спектральная мера

In [ ]:
def spectral_slope_fft(x):
    x = pd.Series(x).dropna().values
    n = len(x)
    if n < 64:
        return np.nan
    x = x - x.mean()
    fft = np.fft.rfft(x)
    psd = np.abs(fft) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    freqs, psd = freqs[1:], psd[1:]  # убрать 0 частоту
    mask = (freqs > 0) & (psd > 0)

    lf = np.log(freqs[mask])
    lp = np.log(psd[mask])

    slope = np.polyfit(lf, lp, 1)[0]
    return slope
measures_roll["spec_slope"] = ts.rolling(WINDOW).apply(spectral_slope_fft, raw=False)
plt.figure()
plt.plot(measures_roll.index, measures_roll["spec_slope"].values)
plt.title(f"Rolling spectral slope (window={WINDOW})")
plt.xlabel("Время")
plt.grid(True)
plt.show()

окно с двигающейся правой границей

In [ ]:
MIN_PERIODS = WINDOW
measures_exp = pd.DataFrame(index=ts.index)
measures_exp["var"]  = ts.expanding(min_periods=MIN_PERIODS).var()
measures_exp["skew"] = ts.expanding(min_periods=MIN_PERIODS).skew()
measures_exp["kurt"] = ts.expanding(min_periods=MIN_PERIODS).kurt()
def acf1_expanding(arr):
    arr = pd.Series(arr).dropna()
    if len(arr) < 3:
        return np.nan
    return arr.autocorr(lag=1)

measures_exp["acf1"] = ts.expanding(min_periods=MIN_PERIODS).apply(acf1_expanding, raw=False)
measures_exp.tail()

херст для стационарных

In [ ]:
# x — ваш ряд (1D массив)
x = ts.dropna().values.astype(float)  # если ts уже есть
T = len(x)

min_s = 16
max_s = T // 2
n_scales = 20

scales = np.unique(np.logspace(np.log10(min_s), np.log10(max_s), n_scales).astype(int))

mu_T = x.mean()
x_centered = x - mu_T

rs_points = []

for s in scales:
    Ns = T // s
    if Ns < 2:
        continue

    rs_vals = []
    for v in range(Ns):
        seg = x_centered[v*s:(v+1)*s]
        seg_mean = seg.mean()

        Y = np.cumsum(seg - seg_mean)   # профиль
        R = Y.max() - Y.min()           # размах
        S = seg.std(ddof=0)             # стандартное отклонение

        if S > 0:
            rs_vals.append(R / S)

    if len(rs_vals) > 0:
        rs_points.append((s, np.mean(rs_vals)))

rs_points = np.array(rs_points)
s_arr = rs_points[:, 0]
F_RS = rs_points[:, 1]

H = np.polyfit(np.log(s_arr), np.log(F_RS), 1)[0]
print("H (R/S) =", H)
plt.figure()
plt.scatter(np.log(s_arr), np.log(F_RS))
plt.title("R/S scaling: log(F_RS) vs log(s)")
plt.xlabel("log(s)")
plt.ylabel("log(F_RS)")
plt.grid(True)
plt.show()

для нестационарных

In [ ]:
x = ts.dropna().values.astype(float)
N = len(x)

min_s = 16
max_s = N // 2
n_scales = 20
order = 1  # 1 = линейный тренд в каждом сегменте

scales = np.unique(np.logspace(np.log10(min_s), np.log10(max_s), n_scales).astype(int))

mu_N = x.mean()
profile = np.cumsum(x - mu_N)  # интегрированный ряд

dfa_points = []

for s in scales:
    Ns = N // s
    if Ns < 2:
        continue

    D_vals = []
    for v in range(Ns):
        seg = profile[v*s:(v+1)*s]
        t = np.arange(s)

        coeffs = np.polyfit(t, seg, order)   # тренд
        trend = np.polyval(coeffs, t)

        detrended = seg - trend
        D = np.mean(detrended**2)            # средний квадрат отклонений
        D_vals.append(D)

    dfa_points.append((s, np.mean(D_vals)))

dfa_points = np.array(dfa_points)
s_arr = dfa_points[:, 0]
F_DFA = dfa_points[:, 1]

H_G = np.polyfit(np.log(s_arr), np.log(F_DFA), 1)[0]
print("H_G (DFA) =", H_G)

plt.figure()
plt.scatter(np.log(s_arr), np.log(F_DFA))
plt.title("DFA scaling: log(F_DFA) vs log(s)")
plt.xlabel("log(s)")
plt.ylabel("log(F_DFA)")
plt.grid(True)
plt.show()

# ML

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay, precision_recall_curve, auc,
    mean_absolute_error, mean_squared_error, r2_score
)

plt.rcParams["figure.figsize"] = (12, 4)
np.random.seed(42)

быстрый обзор данных

In [ ]:
print("размер:", df.shape)
display(df.describe(include="all").T.head(30))
print("пропущенные значения:")
display(df.isna().sum().sort_values(ascending=False).head(20))
print("тайпы:")
display(df.dtypes.value_counts())

общий таргет

In [ ]:
TARGET_COL = "target"   #поменять
DROP_COLS = []          # id если есть
df_model = df.drop(columns=DROP_COLS).copy()
y = df_model[TARGET_COL]
X = df_model.drop(columns=[TARGET_COL])
print("X shape:", X.shape, "| y shape:", y.shape)
print("y dtype:", y.dtype)
print("y unique (first 20):", pd.Series(y).dropna().unique()[:20])

регрессия

In [ ]:
from sklearn.model_selection import train_test_split
TEST_SIZE = 0.2
VAL_SIZE = 0.2
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VAL_SIZE, random_state=42
)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

baseline

In [ ]:
baseline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DummyRegressor(strategy="mean"))
])
baseline.fit(X_train, y_train)
pred_val = baseline.predict(X_val)
print("Baseline MAE :", mean_absolute_error(y_val, pred_val))
print("Baseline RMSE:", mean_squared_error(y_val, pred_val, squared=False))
print("Baseline R2  :", r2_score(y_val, pred_val))

3 модели

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=1.0)": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42)
}
results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = mean_squared_error(y_val, pred, squared=False)
    r2 = r2_score(y_val, pred)

    results.append([name, mae, rmse, r2])
res_df = pd.DataFrame(results, columns=["model", "MAE", "RMSE", "R2"]).sort_values("MAE")
res_df

выбираем из трех моделей

In [ ]:
best_name = res_df.iloc[0]["model"]
print("Best model:", best_name)
best_model = models[best_name]
best_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", best_model)])
best_pipe.fit(X_train_val, y_train_val)
pred_test = best_pipe.predict(X_test)
print("TEST MAE :", mean_absolute_error(y_test, pred_test))
print("TEST RMSE:", mean_squared_error(y_test, pred_test, squared=False))
print("TEST R2  :", r2_score(y_test, pred_test))
# график факт vs прогноз
plt.figure()
plt.scatter(y_test, pred_test)
plt.title("Regression: y_true vs y_pred")
plt.xlabel("y_true")
plt.ylabel("y_pred")
plt.grid(True)
plt.show()
# остатки
residuals = y_test - pred_test
plt.figure()
plt.scatter(pred_test, residuals)
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.grid(True)
plt.show()
plt.figure()
plt.hist(residuals, bins=30)
plt.title("Residuals histogram")
plt.grid(True)
plt.show()

кластеризация

In [ ]:
X_clust = X.copy()
num_cols = X_clust.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_clust.columns if c not in num_cols]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess_clust = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)
X_prepared = preprocess_clust.fit_transform(X_clust)
print("Prepared shape:", X_prepared.shape)

In [ ]:
X_clust = X.copy()
num_cols = X_clust.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_clust.columns if c not in num_cols]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess_clust = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)
X_prepared = preprocess_clust.fit_transform(X_clust)
print("Prepared shape:", X_prepared.shape)

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
clusters = km.fit_predict(X_prepared)
plt.figure()
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, s=10)
plt.title(f"KMeans clusters (k={best_k}) on PCA(2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()
df_clusters = df_model.copy()
df_clusters["cluster"] = clusters
df_clusters[["cluster"]].head()

# Стат тесты

т-тест (равенство средних)

1. сначала проверим нормальность распределения)

In [ ]:
stat, p = st.shapiro(x)
print(stat, p)

2. гомогенность дисперсий

In [ ]:
stat, p = st.levene(a, b)
print(stat, p)

 One-sample t-test

Когда: сравнить среднее выборки с заданным числом m_0.
H0: m = m_0
Условия: независимость, (примерно) нормальность или n большое.


In [ ]:
x = np.array([...])
mu0 = 0

t_stat, p_val = st.ttest_1samp(x, popmean=mu0)
print("t=", t_stat, "p=", p_val)

 Two-sample t-test (независимые)

Когда: сравнить средние двух независимых групп.
H0: \mu_A = \mu_B
Условия: независимость; нормальность (или n большое).
Если дисперсии разные → Welch t-test.


In [ ]:
a = np.array([...])
b = np.array([...])

t_stat, p_val = st.ttest_ind(a, b, equal_var=False)  # Welch
print("t=", t_stat, "p=", p_val)

Paired t-test (парный)

Когда: одни и те же объекты “до/после”.
H0: средняя разность = 0
Условия: нормальность разностей (или n большое).


In [ ]:
before = np.array([...])
after  = np.array([...])

t_stat, p_val = st.ttest_rel(before, after)
print("t=", t_stat, "p=", p_val)

Mann–Whitney U (независимые)

Когда: две независимые группы, данные не нормальные/с выбросами.
H0: распределения одинаковые (часто интерпретируют как “нет сдвига”).
Условия: независимость.

In [ ]:
a = np.array([...])
b = np.array([...])

u_stat, p_val = st.mannwhitneyu(a, b, alternative="two-sided")
print("U=", u_stat, "p=", p_val)

Wilcoxon (парные)

Когда: парное сравнение “до/после”, но разности не нормальны.
H0: медиана разностей = 0.

In [ ]:
before = np.array([...])
after  = np.array([...])

w_stat, p_val = st.wilcoxon(before, after)
print("W=", w_stat, "p=", p_val)

χ² тест независимости

Когда: две категориальные переменные (таблица сопряженности).
H0: независимы.
Условия: ожидаемые частоты не слишком маленькие (часто хотят ≥ 5).

In [ ]:
from scipy.stats import chi2_contingency

# таблица (строки=группы, столбцы=категории)
table = np.array([
    [30, 20],
    [10, 40]
])

chi2, p, dof, expected = chi2_contingency(table)
print("chi2=", chi2, "p=", p, "dof=", dof)
print("expected:\n", expected)

 Z-тест для долей (пропорций)

Когда: сравнить конверсии/доли: p_A vs p_B (например, A/B тест).
H0: p_A = p_B
Условия: достаточно большие n (чтобы нормальная аппроксимация работала).

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# successes: сколько успехов в каждой группе
# nobs: размер каждой группы
successes = np.array([50, 65])
nobs = np.array([200, 210])

z_stat, p_val = proportions_ztest(count=successes, nobs=nobs, alternative="two-sided")
print("z=", z_stat, "p=", p_val)

Pearson

Когда: связь линейная, данные примерно нормальные без сильных выбросов.
H0: корреляция = 0.

In [ ]:
x = np.array([...])
y = np.array([...])

r, p = st.pearsonr(x, y)
print("Pearson r=", r, "p=", p)

Spearman

Когда: связь монотонная, данные могут быть не нормальные/ранговые.
H0: корреляция = 0.

In [ ]:
rho, p = st.spearmanr(x, y)
print("Spearman rho=", rho, "p=", p)

Доверительный интервал для среднего (самый классический)

3.1 Если σ неизвестна (обычно так) → t-интервал

In [ ]:
import numpy as np
import scipy.stats as st

x = np.array([...])
n = len(x)
mean = x.mean()
s = x.std(ddof=1)

alpha = 0.05
t_crit = st.t.ppf(1 - alpha/2, df=n-1)
se = s / np.sqrt(n)

ci = (mean - t_crit*se, mean + t_crit*se)
ci

Доверительный интервал для доли (конверсии)

In [ ]:
import numpy as np
import scipy.stats as st

x = 50
n = 200
p_hat = x / n

alpha = 0.05
z = st.norm.ppf(1 - alpha/2)
se = np.sqrt(p_hat*(1-p_hat)/n)

ci = (p_hat - z*se, p_hat + z*se)
ci

Доверительный интервал для разницы долей (A/B тест)

In [ ]:
import numpy as np
import scipy.stats as st

x1, n1 = 50, 200
x2, n2 = 65, 210

p1 = x1/n1
p2 = x2/n2

diff = p1 - p2

alpha = 0.05
z = st.norm.ppf(1 - alpha/2)

se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)

ci = (diff - z*se, diff + z*se)
diff, ci

# AB тест

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.stats as st

from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

df = pd.read_csv("your_ab_data.csv")
df.head()

Дизайн эксперимента

In [ ]:
#Гипотеза
hypothesis = """
Если мы изменим X (фича/дизайн/алгоритм), то метрика metric_main вырастет,
потому что пользователям станет проще/быстрее/понятнее.
"""

# Основная метрика и guard-rail метрики
main_metric = "metric_main"           # например: "converted" (0/1) или "revenue"
guardrail_metrics = []               # например: ["errors", "refunds", "load_time"]

#Параметры статистики
alpha = 0.05                          # уровень значимости (ошибка I рода)
power_target = 0.80                   # желаемая мощность

# 4) MDE
mde_relative = 0.02                   # пример: хотим уметь ловить +2% относительного эффекта
print(hypothesis)

готовим данные

In [ ]:
# Убираем пропуски в основной метрике
df = df.dropna(subset=[main_metric]).copy()

print("Размер df:", df.shape)
print(df[main_metric].describe())

Проверка уникальности пользователей

In [ ]:
dup_users = df.duplicated(subset=["user_id"]).sum()
print("Дубликаты user_id:", dup_users)

сплит

In [ ]:
if "group" not in df.columns:
    # стабильный псевдо-случайный сплит через хэш (важно: один и тот же user_id всегда попадает в ту же группу)
    h = pd.util.hash_pandas_object(df["user_id"], index=False).astype("uint64")
    bucket = (h % 100).astype(int)
    df["group"] = np.where(bucket < 50, "A", "B")

df["group"].value_counts()

проверка хи квадратом похожести групп

In [ ]:
cat_col = None  # например: "device"
if cat_col is not None and cat_col in df.columns:
    ct = pd.crosstab(df["group"], df[cat_col])
    display(ct)

    chi2, p, dof, expected = st.chi2_contingency(ct)
    print("Chi2 p-value:", p)

АА тест

In [ ]:
#выбираем метрику
aa_metric = main_metric
x = df[aa_metric].values
n = len(x)

прогоны АА

In [ ]:
np.random.seed(42)
B = 500  # число повторов A/A
pvals = []
is_binary = set(pd.unique(df[aa_metric])) <= {0, 1}
for _ in range(B):
    mask = np.random.rand(n) < 0.5
    a = x[mask]
    b = x[~mask]

    if is_binary:
        # z-test пропорций
        x1, n1 = a.sum(), len(a)
        x2, n2 = b.sum(), len(b)
        _, p = proportions_ztest([x1, x2], [n1, n2], alternative="two-sided")
    else:
        # Welch t-test
        _, p = st.ttest_ind(a, b, equal_var=False)
    pvals.append(p)
pvals = np.array(pvals)
print("Доля p-value < 0.05:", (pvals < 0.05).mean())

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(pvals, bins=20)
plt.title("A/A: распределение p-value (должно быть близко к равномерному)")
plt.xlabel("p-value")
plt.ylabel("count")
plt.show()

In [ ]:
# базовая конверсия (оценка по историческим данным / контролю)
if is_binary:
    p_baseline = df[main_metric].mean()
    p_target = p_baseline * (1 + mde_relative)  # хотим +mde_relative

    effect = proportion_effectsize(p_baseline, p_target)
    analysis = NormalIndPower()
    n_per_group = int(np.ceil(analysis.solve_power(effect_size=effect, power=power_target, alpha=alpha, ratio=1.0)))

    print("Baseline p:", p_baseline)
    print("Target p:", p_target)
    print("Нужно примерно наблюдений на группу:", n_per_group)

In [ ]:
if not is_binary:
    # грубая оценка через std и минимальный абсолютный эффект
    mu = df[main_metric].mean()
    sigma = df[main_metric].std(ddof=1)
    delta = mu * mde_relative  # хотим сдвиг среднего на mde_relative

    effect = delta / sigma  # Cohen's d
    analysis = TTestIndPower()
    n_per_group = int(np.ceil(analysis.solve_power(effect_size=effect, power=power_target, alpha=alpha, ratio=1.0)))

    print("Mean:", mu, "Std:", sigma)
    print("Delta (abs):", delta, "Effect size (d):", effect)
    print("Нужно примерно наблюдений на группу:", n_per_group)

Длительность теста (проверка недели)

In [ ]:
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    days = (df["date"].max() - df["date"].min()).days + 1
    print("Длительность данных (дней):", days)

Разделяем на группы

In [ ]:
A = df.loc[df["group"] == "A", main_metric].values
B = df.loc[df["group"] == "B", main_metric].values
print("nA:", len(A), "nB:", len(B))

In [ ]:
Сценарий 1: конверсия → z-test пропорций + CI

In [ ]:
if is_binary:
    x1, n1 = A.sum(), len(A)
    x2, n2 = B.sum(), len(B)

    p1 = x1 / n1
    p2 = x2 / n2
    diff = p2 - p1
    uplift = diff / p1

    z_stat, p_value = proportions_ztest([x1, x2], [n1, n2], alternative="two-sided")

    # CI для разницы долей (норм. приближение)
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    z_crit = st.norm.ppf(1 - alpha/2)
    ci = (diff - z_crit*se, diff + z_crit*se)

    print(f"pA={p1:.4f}  pB={p2:.4f}")
    print(f"diff={diff:.4f}  uplift={uplift:.2%}")
    print(f"z={z_stat:.3f}  p-value={p_value:.4f}")
    print(f"95% CI for (pB - pA): [{ci[0]:.4f}, {ci[1]:.4f}]")

Сценарий 2: средний чек/время → Welch t-test + CI

In [ ]:
if not is_binary:
    meanA, meanB = A.mean(), B.mean()
    diff = meanB - meanA
    uplift = diff / meanA

    t_stat, p_value = st.ttest_ind(A, B, equal_var=False)

    # Приближённый CI (норм. аппрокс. для diff)
    se = np.sqrt(A.var(ddof=1)/len(A) + B.var(ddof=1)/len(B))
    z_crit = st.norm.ppf(1 - alpha/2)
    ci = (diff - z_crit*se, diff + z_crit*se)

    print(f"meanA={meanA:.4f}  meanB={meanB:.4f}")
    print(f"diff={diff:.4f}  uplift={uplift:.2%}")
    print(f"t={t_stat:.3f}  p-value={p_value:.4f}")
    print(f"~95% CI for (meanB - meanA): [{ci[0]:.4f}, {ci[1]:.4f}]")

 Принятие решения

In [ ]:
mde_abs = mde_relative  # если метрика относительная (конверсия/среднее), удобно сравнивать uplift с mde_relative
print("alpha =", alpha)
print("MDE (relative) =", mde_relative)
if p_value < alpha and uplift >= mde_relative:
    decision = "Значимый положительный эффект и uplift ≥ MDE: внедряем (после проверки guard-rails)."
elif p_value < alpha and uplift < mde_relative:
    decision = "Значимо, но эффект < MDE: эффект слабый — решаем с учётом ресурсов/ценности."
elif p_value >= alpha:
    decision = "Не значимо: нет уверенности — можно увеличить выборку/длительность или закрыть тест."
else:
    decision = "Проверь расчёты/знаки эффекта."

print(decision)

# EDA

общая инфа

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)

пропуски

In [ ]:
na_cnt = df.isna().sum()
na_share = (na_cnt / len(df)).sort_values(ascending=False)

na_table = pd.DataFrame({
    "missing_cnt": na_cnt,
    "missing_share": (na_cnt / len(df))
}).sort_values("missing_share", ascending=False)
display(na_table.head(30))
plt.figure()
plt.hist(na_table["missing_share"], bins=30)
plt.title("Distribution of missing share across columns")
plt.xlabel("missing_share")
plt.ylabel("count of columns")
plt.grid(True)
plt.show()

дубликаты

In [ ]:
print("Duplicated rows:", df.duplicated().sum())
# если есть id — проверь уникальность
ID_COL = None  # например "id"
if ID_COL is not None and ID_COL in df.columns:
    print("Unique IDs:", df[ID_COL].nunique(), "of", len(df))

типы колонок

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Numeric:", len(num_cols))
print("Categorical/bool:", len(cat_cols))
# Попытка автоматически найти дату по названию
date_like = [c for c in df.columns if any(k in c.lower() for k in ["date", "time", "dt", "timestamp"])]
print("Date-like columns:", date_like)

удалить колонку

In [ ]:
df = df.drop(columns=["col_name"])

удалить пропуски

In [ ]:
df_clean = df.dropna() #где есть хотя бы один
df_clean = df.dropna(subset=["target", "age", "income"])

фильтр

In [ ]:
df_filt = df[df["age"] > 30]
df_filt = df[(df["age"] > 30) & (df["city"] == "Vienna")]

#по списку значений
df_filt = df[df["city"].isin(["Vienna", "Graz", "Linz"])]

#по строкам
df_filt = df[df["name"].str.contains("anna", case=False, na=False)]

#по пропуску
df_filt = df[df["age"].isna()]      # где NaN
df_filt = df[df["age"].notna()]     # где НЕ NaN

#по диапазону
df_filt = df[df["age"].between(18, 30)]

df["date"] = pd.to_datetime(df["date"])

df_filt = df[df["date"] >= "2024-01-01"]
# или диапазон
df_filt = df[df["date"].between("2024-01-01", "2024-12-31")]

приведение дат

In [ ]:
DATE_COL = None  # например "date"
if DATE_COL is not None and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    print("Converted to datetime. Missing after parse:", df[DATE_COL].isna().sum())

анализ чисел

In [ ]:
if len(num_cols) > 0:
    desc_num = df[num_cols].describe().T
    desc_num["missing_cnt"] = df[num_cols].isna().sum()
    desc_num["missing_share"] = desc_num["missing_cnt"] / len(df)
    display(desc_num.sort_values("missing_share", ascending=False).head(30))

гистограмма для числовых

In [ ]:
N = 8
cols_to_plot = num_cols[:N]

for c in cols_to_plot:
    plt.figure()
    plt.hist(df[c].dropna(), bins=40)
    plt.title(f"Histogram: {c}")
    plt.xlabel(c)
    plt.ylabel("count")
    plt.grid(True)
    plt.show()

boxplot для числовых

In [ ]:
for c in cols_to_plot:
    plt.figure()
    plt.boxplot(df[c].dropna(), vert=False, showfliers=True)
    plt.title(f"Boxplot: {c}")
    plt.xlabel(c)
    plt.grid(True)
    plt.show()

поиск выбросов

In [ ]:
outliers = []
for c in num_cols:
    x = df[c].dropna()
    if len(x) < 10:
        continue
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    cnt = ((x < low) | (x > high)).sum()
    outliers.append([c, cnt, cnt/len(x), low, high])
out_df = pd.DataFrame(outliers, columns=["col", "out_cnt", "out_share", "low", "high"]).sort_values("out_share", ascending=False)
display(out_df.head(30))

анализ категориальных

топ значений

In [ ]:
N = 5
for c in cat_cols[:N]:
    print(f"\n=== {c} ===")
    display(df[c].value_counts(dropna=False).head(15))

Барчарты по категориям

In [ ]:
for c in cat_cols[:N]:
    vc = df[c].value_counts(dropna=False)
    if len(vc) <= 20:
        plt.figure()
        plt.bar(vc.index.astype(str), vc.values)
        plt.title(f"Bar chart: {c}")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y")
        plt.show()

связь

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    # топ пар по модулю корреляции
    corr_pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            .stack()
            .reset_index()
    )
    corr_pairs.columns = ["col1", "col2", "corr"]
    corr_pairs["abs_corr"] = corr_pairs["corr"].abs()

    display(corr_pairs.sort_values("abs_corr", ascending=False).head(20))

Heatmap корреляций

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    plt.imshow(corr.values)
    plt.title("Correlation heatmap (numeric)")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.colorbar()
    plt.grid(False)
    plt.show()

scatter для пары признаков

In [ ]:
X_COL = None  # например "age"
Y_COL = None  # например "income"
if X_COL is not None and Y_COL is not None:
    plt.figure()
    plt.scatter(df[X_COL], df[Y_COL], s=10)
    plt.title(f"Scatter: {X_COL} vs {Y_COL}")
    plt.xlabel(X_COL)
    plt.ylabel(Y_COL)
    plt.grid(True)
    plt.show()

итоговый список колонок

In [ ]:
print("NUMERIC COLS:")
print(num_cols)
print("\nCATEGORICAL COLS:")
print(cat_cols)

# визуализации

Гистограмма (Histogram)

Что показывает: распределение числового признака по интервалам (сколько значений попало в каждый диапазон).
Зачем нужна:

	•	понять форму распределения (нормальное/скошенное/мультимодальное)
	•	увидеть “длинный хвост”, асимметрию
	•	заметить выбросы и аномальные пики
Когда: любой числовой признак (цены, время, возраст, продажи).

In [ ]:
NUM = "age"
plt.figure()
plt.hist(df[NUM].dropna(), bins=40)
plt.title(f"Histogram: {NUM}")
plt.grid(True)
plt.show()

Boxplot (ящик с усами)

Что показывает: медиану, квартили (Q1–Q3), “усы” и выбросы.
Зачем нужна:

	•	быстро увидеть разброс и выбросы
	•	сравнить распределения между группами (по категориям)
Когда: сравнение числовой метрики по группам (например, доход по городам).

In [ ]:
NUM = "income"
plt.figure()
plt.boxplot(df[NUM].dropna(), vert=False, showfliers=True)
plt.title(f"Boxplot: {NUM}")
plt.grid(True)
plt.show()

Violin plot (скрипка)

Что показывает: сочетание boxplot + форма распределения (плотность).
Зачем нужна:

	•	видеть и статистику (медиана/квартили), и “форму” (где плотнее значения)
	•	полезно, если boxplot скрывает детали распределения
Когда: сравнение метрики между группами, особенно если распределение сложное.

In [ ]:
CAT = "group"
NUM = "revenue"
groups = df[[CAT, NUM]].dropna().groupby(CAT)[NUM].apply(list)
plt.figure(figsize=(12,6))
plt.violinplot(groups.values, showmeans=True, showmedians=True)
plt.xticks(range(1, len(groups)+1), groups.index.astype(str), rotation=45)
plt.title(f"Violin: {NUM} by {CAT}")
plt.grid(True)
plt.show()

Столбчатая диаграмма (Bar chart)

Что показывает: значения категорий (количество, доля, среднее).
Зачем нужна:

	•	распределение категорий (топ городов, топ устройств)
	•	сравнение метрики по категориям (mean revenue по сегментам)
Когда: категориальные признаки или агрегаты по группам.

In [ ]:
CAT = "city"
vc = df[CAT].value_counts().head(15)
plt.figure()
plt.bar(vc.index.astype(str), vc.values)
plt.title(f"Top categories: {CAT}")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.show()

Линейный график (Line plot)

Что показывает: изменение значения во времени/по порядку.
Зачем нужна:

	•	тренды, сезонность, “скачки”
	•	визуально оценить стабильность метрики (например, в A/B тесте по дням)
	•	выявить структурные сдвиги
Когда: временные ряды, метрика по времени, накопительный эффект.

In [ ]:
DATE = "date"
NUM = "revenue"
tmp = df[[DATE, NUM]].dropna().copy()
tmp[DATE] = pd.to_datetime(tmp[DATE], errors="coerce")
daily = tmp.groupby(tmp[DATE].dt.date)[NUM].mean()
plt.figure()
plt.plot(daily.index, daily.values)
plt.title(f"Daily mean of {NUM}")
plt.grid(True)
plt.show()

Scatter plot (диаграмма рассеяния)

Что показывает: связь двух числовых признаков (точки на плоскости).
Зачем нужна:

	•	увидеть корреляцию (линейную/нелинейную)
	•	увидеть кластеры
	•	увидеть выбросы (точки далеко от облака)
Когда: “число vs число” (возраст vs доход, цена vs площадь).


In [ ]:
NUM = "age"
NUM2 = "income"
plt.figure()
plt.scatter(df[NUM], df[NUM2], s=10)
plt.title(f"Scatter: {NUM} vs {NUM2}")
plt.xlabel(NUM)
plt.ylabel(NUM2)
plt.grid(True)
plt.show()

Heatmap (тепловая карта)

Что показывает: матрицу значений цветом (часто корреляции).
Зачем нужна:

	•	быстро увидеть сильные связи между признаками
	•	искать мультиколлинеарность (важно для линейных моделей)
Когда: корреляционная матрица, таблицы частот (например, “день недели × час”).

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns
corr = df[num_cols].corr(numeric_only=True)
plt.figure(figsize=(10,8))
plt.imshow(corr.values)
plt.title("Correlation heatmap")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.grid(False)
plt.show()

QQ-plot

Что показывает: насколько данные похожи на нормальное распределение.
Зачем нужна:

	•	визуальная проверка нормальности (для t-test, регрессии, остатков)
Когда: проверяешь нормальность данных или остатков модели.

In [ ]:
NUM = "income"
x = df[NUM].dropna().values
plt.figure(figsize=(5,5))
st.probplot(x, dist="norm", plot=plt)
plt.title(f"QQ-plot: {NUM}")
plt.grid(True)
plt.show()

# основной раздел

In [ ]:
#НАМПАЙ
# создание массивов
a = np.array([1, 2, 3])              # из списка
b = np.array([[1, 2], [3, 4]])       # 2D массив (матрица)

z = np.zeros((2, 3))                 # массив нулей
o = np.ones((2, 3))                  # массив единиц
e = np.empty((2, 3))                 # пустой
f = np.full((2, 3), 7)               # заполнить одним значением

r = np.arange(0, 10, 2)                 # [0, 2, 4, 6, 8]
lin = np.linspace(0, 1, 5)              # 5 точек от 0 до 1 включительно

eye = np.eye(3)                         # единичная матрица 3x3
I = np.identity(3)                      # то же самое

rand = np.random.rand(3, 4)             # равномерное [0,1)
randn = np.random.randn(3, 4)           # нормальное N(0,1)
randi = np.random.randint(0, 10, (3, 4))# целые случайные


# атрибуты массива
arr = np.array([[1, 2, 3], [4, 5, 6]])
arr.shape                            # форма (строки, столбцы)
arr.ndim                             # размерность
arr.size                             # количество элементов
arr.dtype                            # тип данных
arr.T                                # транспонирование (2D)


# изменение формы
arr.reshape(3, 2)                    # изменить shape
arr.reshape(-1, 1)                   # -1 = "досчитай сам"
arr.ravel()                          # в 1D (копия/вид)
arr.flatten()                        # в 1D (точно копия)

np.concatenate([a, a])                  # склеить по оси
np.stack([a, a], axis=0)                # "сложить" по новой оси
np.hstack([a, a])                       # горизонтально
np.vstack([a, a])                       # вертикально
np.split(r, 2)                          # разрезать
np.hsplit(np.array([[1,2],[3,4]]), 2)   # split по столбцам
np.vsplit(np.array([[1,2],[3,4]]), 2)   # split по строкам


# индексация и срезы
arr[0, 1]                               # элемент
arr[0, :]                               # строка
arr[:, 1]                               # столбец
arr[:1, :2]                             # срез подматрицы
arr[::2, :]                             # шаг

mask = arr > 3                          # булева маска
arr[mask]                               # фильтрация по условию
np.where(arr > 3, 1, 0)                 # if-else поэлементно

idx = np.argmax(arr)                    # индекс максимума (в развернутом виде)
idx2 = np.unravel_index(np.argmax(arr), arr.shape)  # индекс в (i,j)


# арифметика и матоперации
x = np.array([1, 2, 3])
y = np.array([10, 20, 30])

x + y                                   # поэлементно
x * 2                                   # умножение на число
x / (y + 1)                             # поэлементно

np.add(x, y)
np.subtract(x, y)
np.multiply(x, y)
np.divide(x, y)

A = np.array([[1, 2], [3, 4]])
B = np.array([[10, 0], [0, 10]])

A @ B                                   # матричное умножение
np.dot(A, B)                            # то же для 2D
np.matmul(A, B)                         # то же
np.linalg.inv(B)                        # обратная матрица (если существует)
np.linalg.det(B)                        # определитель


# агрегации и статистика
m = np.array([[1, 2, 3], [4, 5, 6]])

m.sum()                                 # сумма всех
m.sum(axis=0)                           # по столбцам
m.sum(axis=1)                           # по строкам

m.mean()
m.min()
m.max()
m.std()
m.var()
np.median(m)
np.quantile(m, 0.75)

np.unique(m)                            # уникальные значения
np.count_nonzero(m)                     # ненулевые

np.argmax(m, axis=0)                    # индексы максимумов по столбцам
np.argmin(m, axis=1)                    # индексы минимумов по строкам


# сортировка
np.sort(x)                              # сортировка (копия)
np.argsort(y)                           # индексы сортировки
np.sort(m, axis=1)                      # сортировка по строкам


# работа с типами -
x.astype(float)                         # привести тип
np.round(np.array([1.234, 2.678]), 2)   # округление

In [ ]:
#ПАНДАС
# создание
s = pd.Series([10, 20, 30], name="vals")
df = pd.DataFrame({
    "A": [1, 2, 3],
    "B": [10, 20, 30],
    "C": ["x", "y", "z"]
})

# загрузка и схранение
df = pd.read_csv("file.csv")
df = pd.read_excel("file.xlsx")
df.to_csv("out.csv", index=False)
df.to_excel("out.xlsx", index=False)

# просмотр
df.head()
df.tail()
df.sample(3)

df.shape
df.columns
df.dtypes
df.info()
df.describe(include="all")              # статистика по всем типам

df.nunique()                            # число уникальных по столбцам
df.value_counts()                       # для Series: s.value_counts()


# выбор данных: столбцы/строки
df["A"]                                 # Series
df[["A", "B"]]                           # DataFrame

df.loc[0]                               # строка по индексу-метке
df.iloc[0]                              # строка по позиции

df.loc[:, ["A", "C"]]                   # столбцы по именам
df.iloc[:, 0:2]                         # столбцы по позициям

df.at[0, "A"]                           # быстро: 1 значение по label
df.iat[0, 0]                            # быстро: 1 значение по position


# фильтрация (маски)
mask = df["A"] > 1
df[mask]

df[(df["A"] > 1) & (df["C"] != "z")]    # несколько условий
df.query("A > 1 and C != 'z'")          # фильтр через строку


# создание/изменение столбцов
df["D"] = df["A"] + df["B"]             # новый столбец
df = df.assign(E=lambda x: x["B"] * 2)  # через assign

df.rename(columns={"A": "A_new"})       # переименовать столбцы
df.rename(index={0: "row0"})            # переименовать индексы

df.drop(columns=["C"])                  # удалить столбцы
df.drop(index=[0])                      # удалить строки


# пропуски
df.isna()
df.isna().sum()

df.dropna()                             # удалить строки с NaN
df.fillna(0)                            # заполнить NaN

# типы данных
df.astype({"A": "int64"})
pd.to_numeric(df["B"], errors="coerce")
pd.to_datetime(pd.Series(["2025-01-01", "2025-01-02"]), errors="coerce")

# сортировка
df.sort_values(by="B", ascending=False)
df.sort_index()

# дубликаты
df.duplicated()
df.drop_duplicates()

# группировки и агрегации
df.groupby("C")["B"].mean()
df.groupby("C").agg(
    mean_B=("B", "mean"),
    sum_B=("B", "sum"),
    cnt=("B", "count")
).reset_index()

# сводные таблицы (pivot)
# pivot — если индекс/колонки уникальны
# df.pivot(index="C", columns="A", values="B")

# pivot_table — если нужны агрегации
df.pivot_table(index="C", values="B", aggfunc="mean")

# объединения (merge / concat / join)
df2 = pd.DataFrame({"A": [1, 2, 3], "X": [100, 200, 300]})
df_merge = df.merge(df2, on="A", how="left")   # left/right/inner/outer

df_concat = pd.concat([df, df], axis=0)        # склеить по строкам
df_concat_cols = pd.concat([df, df2], axis=1)  # склеить по столбцам

# применение функций
df["B"].apply(lambda v: v * 10)          # по Series
df.apply(lambda row: row["A"] + row["B"], axis=1)  # по строкам (медленнее)

# строковые методы
df["C"].str.lower()
df["C"].str.strip()
df["C"].str.contains("x")

# дата время
dates = pd.to_datetime(pd.Series(["2025-12-28 10:00", "2025-12-29 12:30"]))
dates.dt.date
dates.dt.day
dates.dt.month
dates.dt.dayofweek
dates.dt.hour
